# 🧬 Enhancer Repair Research: AI-Driven Regulatory Grammar Discovery

**Decoding Regulatory Grammar Through Neural Network Repair**

This notebook implements a novel methodology for discovering gene regulatory grammar by using neural network "repair" as a biological probe.

## Research Questions
1. **Cross-Species Conservation:** Which regulatory motifs are conserved across vertebrate evolution?
2. **Combinatorial Grammar:** Which motifs work synergistically versus independently?
3. **Generative Understanding:** Can the model create functional enhancers from scratch?

---
**Important:** Run cells in order. Use `Runtime > Run all` for full execution.

## 1. Environment Setup

### 1.1 Check GPU Availability

In [ ]:
# Check if running in Colab
try:
    import google.colab
    IN_COLAB = True
    print("Running in Google Colab")
except ImportError:
    IN_COLAB = False
    print("Running locally")

# Check GPU
import torch
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    device = torch.device('cuda')
else:
    print("WARNING: No GPU detected. Training will be slow.")
    device = torch.device('cpu')

print(f"\nUsing device: {device}")

### 1.2 Mount Google Drive (for persistent storage)

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Create project directory
    PROJECT_DIR = '/content/drive/MyDrive/enhancer_repair_research'
else:
    PROJECT_DIR = '.'

import os
from pathlib import Path

# Create directory structure
dirs = ['data/raw', 'data/processed', 'models', 'checkpoints', 
        'results/cross_species', 'results/grammar', 'results/generative',
        'figures', 'logs']

for d in dirs:
    Path(f"{PROJECT_DIR}/{d}").mkdir(parents=True, exist_ok=True)

print(f"Project directory: {PROJECT_DIR}")
print("Directory structure created.")

### 1.3 Install Dependencies

In [ ]:
if IN_COLAB:
    print("Installing additional dependencies...")
    !pip install -q biopython==1.81 captum==0.6.0
    print("Dependencies installed.")

# Verify imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import pickle
import json
import time
import warnings
warnings.filterwarnings('ignore')

print("All imports successful!")

## 2. Configuration

Optimized for Google Colab T4 GPU (16GB VRAM)

In [ ]:
# Training Configuration
CONFIG = {
    # Model Architecture
    'n_transformer_layers': 10,
    'n_attention_heads': 8,
    'd_model': 256,
    'dim_feedforward': 1536,
    'dropout': 0.1,
    
    # Training
    'batch_size': 64,
    'num_epochs': 30,
    'learning_rate': 0.0001,
    'weight_decay': 0.01,
    
    # Optimization
    'gradient_accumulation_steps': 4,
    'max_grad_norm': 0.5,
    
    # Scheduling
    'scheduler_factor': 0.5,
    'scheduler_patience': 3,
    'scheduler_min_lr': 1e-6,
    
    # Early stopping
    'early_stopping_patience': 7,
    
    # Stochastic sampling (memory efficiency)
    'samples_per_epoch': 100000,
    
    # Mixed precision
    'use_mixed_precision': True,
    
    # Data loading
    'num_workers': 2,
    'pin_memory': True,
    
    # Paths
    'project_dir': PROJECT_DIR,
    'checkpoint_dir': f"{PROJECT_DIR}/checkpoints",
    'data_dir': f"{PROJECT_DIR}/data/processed",
    'figures_dir': f"{PROJECT_DIR}/figures",
    'results_dir': f"{PROJECT_DIR}/results",
}

# Cross-species analysis config
CROSS_SPECIES_CONFIG = {
    'n_sequences_per_species': 300,
    'damage_fraction': 0.1,
    'max_repair_iterations': 30,
    'repair_learning_rate': 0.01,
    'target_probability': 0.9,
}

# Grammar discovery config
GRAMMAR_CONFIG = {
    'n_pairs_to_test': 500,
    'n_sequences_per_pair': 3,
    'n_sequences_for_motifs': 50,
    'importance_percentile': 75,
}

# Generative validation config
GENERATIVE_CONFIG = {
    'n_generation_attempts': 200,
    'max_iterations': 500,
    'target_probability': 0.80,
    'learning_rate': 0.05,
}

print("Configuration set!")
print(f"Project directory: {CONFIG['project_dir']}")

## 3. Data Preparation

### 3.1 DNA Encoding Functions

In [ ]:
def one_hot_encode(sequence):
    """Convert DNA string to one-hot encoded array (4, seq_len)."""
    mapping = {'A': 0, 'C': 1, 'G': 2, 'T': 3}
    seq_len = len(sequence)
    encoded = np.zeros((4, seq_len), dtype=np.float32)
    for i, nucleotide in enumerate(sequence.upper()):
        if nucleotide in mapping:
            encoded[mapping[nucleotide], i] = 1.0
    return encoded

def decode_sequence(one_hot):
    """Convert one-hot array back to DNA string."""
    mapping = {0: 'A', 1: 'C', 2: 'G', 3: 'T'}
    seq_len = one_hot.shape[1]
    return ''.join([mapping[np.argmax(one_hot[:, i])] for i in range(seq_len)])

# Test
test_seq = "ATGCGTACGATCGATCGTAGCTAGCTACGATCGATCG"
encoded = one_hot_encode(test_seq)
decoded = decode_sequence(encoded)
print(f"Original: {test_seq}")
print(f"Decoded:  {decoded}")
print(f"Match: {test_seq == decoded}")

### 3.2 Generate Synthetic Data (Demo Mode)

For demo purposes, we'll generate synthetic data that mimics real enhancer properties.
For full research, replace this with real ENCODE data download.

In [ ]:
import random

def generate_synthetic_enhancer(length=200):
    """Generate synthetic enhancer-like sequence with motifs."""
    # Known motifs that will make sequences more "enhancer-like"
    motifs = ['TATAAA', 'CACGTG', 'GGGCGG', 'CCAAT', 'GCGCGC']
    
    # Start with random sequence (higher GC content like real enhancers)
    gc_prob = 0.45
    nucleotides = ['A', 'C', 'G', 'T']
    probs = [(1-gc_prob)/2, gc_prob/2, gc_prob/2, (1-gc_prob)/2]
    seq = ''.join(random.choices(nucleotides, weights=probs, k=length))
    seq = list(seq)
    
    # Insert 2-4 motifs at random positions
    n_motifs = random.randint(2, 4)
    for _ in range(n_motifs):
        motif = random.choice(motifs)
        pos = random.randint(10, length - len(motif) - 10)
        for i, nt in enumerate(motif):
            seq[pos + i] = nt
    
    return ''.join(seq)

def generate_background_sequence(length=200):
    """Generate random background sequence."""
    gc_prob = 0.42  # Genome average
    nucleotides = ['A', 'C', 'G', 'T']
    probs = [(1-gc_prob)/2, gc_prob/2, gc_prob/2, (1-gc_prob)/2]
    return ''.join(random.choices(nucleotides, weights=probs, k=length))

# Generate synthetic dataset
np.random.seed(42)
random.seed(42)

# Dataset sizes (reduced for demo - increase for full research)
N_HUMAN = 20000
N_MOUSE = 15000
N_ZEBRAFISH = 10000
N_CHICKEN = 10000
N_BACKGROUND = 55000  # Match total enhancers

print("Generating synthetic dataset...")
print(f"  Human enhancers: {N_HUMAN:,}")
print(f"  Mouse enhancers: {N_MOUSE:,}")
print(f"  Zebrafish enhancers: {N_ZEBRAFISH:,}")
print(f"  Chicken enhancers: {N_CHICKEN:,}")
print(f"  Background: {N_BACKGROUND:,}")

human_seqs = [generate_synthetic_enhancer() for _ in tqdm(range(N_HUMAN), desc="Human")]
mouse_seqs = [generate_synthetic_enhancer() for _ in tqdm(range(N_MOUSE), desc="Mouse")]
zebrafish_seqs = [generate_synthetic_enhancer() for _ in tqdm(range(N_ZEBRAFISH), desc="Zebrafish")]
chicken_seqs = [generate_synthetic_enhancer() for _ in tqdm(range(N_CHICKEN), desc="Chicken")]
background_seqs = [generate_background_sequence() for _ in tqdm(range(N_BACKGROUND), desc="Background")]

print(f"\nTotal enhancers: {N_HUMAN + N_MOUSE + N_ZEBRAFISH + N_CHICKEN:,}")
print(f"Total background: {N_BACKGROUND:,}")

### 3.3 Prepare Dataset

In [ ]:
from sklearn.model_selection import train_test_split

print("Preparing dataset...")

# Combine all sequences
all_sequences = []
all_labels = []
species_labels = []

# Human (species_id=0)
all_sequences.extend(human_seqs)
all_labels.extend([1] * len(human_seqs))
species_labels.extend([0] * len(human_seqs))

# Mouse (species_id=1)
all_sequences.extend(mouse_seqs)
all_labels.extend([1] * len(mouse_seqs))
species_labels.extend([1] * len(mouse_seqs))

# Zebrafish (species_id=2)
all_sequences.extend(zebrafish_seqs)
all_labels.extend([1] * len(zebrafish_seqs))
species_labels.extend([2] * len(zebrafish_seqs))

# Chicken (species_id=3)
all_sequences.extend(chicken_seqs)
all_labels.extend([1] * len(chicken_seqs))
species_labels.extend([3] * len(chicken_seqs))

# Background (species_id=4)
all_sequences.extend(background_seqs)
all_labels.extend([0] * len(background_seqs))
species_labels.extend([4] * len(background_seqs))

print(f"Total sequences: {len(all_sequences):,}")

# One-hot encode
print("\nOne-hot encoding...")
X = np.array([one_hot_encode(seq) for seq in tqdm(all_sequences, desc="Encoding")])
y = np.array(all_labels, dtype=np.int64)
species = np.array(species_labels, dtype=np.int64)

print(f"Dataset shape: X={X.shape}, y={y.shape}")

# Create stratified splits
print("\nCreating train/val/test splits...")
indices = np.arange(len(y))

train_val_idx, test_idx = train_test_split(
    indices, test_size=0.15, stratify=y, random_state=42
)
train_idx, val_idx = train_test_split(
    train_val_idx, test_size=0.176, stratify=y[train_val_idx], random_state=42
)

print(f"Train: {len(train_idx):,} ({len(train_idx)/len(y)*100:.1f}%)")
print(f"Val: {len(val_idx):,} ({len(val_idx)/len(y)*100:.1f}%)")
print(f"Test: {len(test_idx):,} ({len(test_idx)/len(y)*100:.1f}%)")

### 3.4 Save Dataset

In [ ]:
# Save processed data
data_dir = Path(CONFIG['data_dir'])
data_dir.mkdir(parents=True, exist_ok=True)

print("Saving dataset...")
np.savez_compressed(
    data_dir / 'dataset.npz',
    X=X, y=y, species=species
)

np.savez(
    data_dir / 'splits.npz',
    train_idx=train_idx, val_idx=val_idx, test_idx=test_idx
)

metadata = {
    'total_sequences': len(X),
    'sequence_length': 200,
    'n_train': len(train_idx),
    'n_val': len(val_idx),
    'n_test': len(test_idx),
    'species_names': ['human', 'mouse', 'zebrafish', 'chicken', 'background']
}

with open(data_dir / 'metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"Dataset saved to {data_dir}")

## 4. Model Architecture

### 4.1 Define Model Components

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import math

class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding."""
    def __init__(self, d_model, max_len=200, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


class CNNEncoder(nn.Module):
    """CNN for local motif detection."""
    def __init__(self, input_channels=4, hidden_channels=[128, 256, 384]):
        super().__init__()
        
        self.conv1 = nn.Conv1d(input_channels, hidden_channels[0], kernel_size=7, padding=3)
        self.bn1 = nn.BatchNorm1d(hidden_channels[0])
        
        self.conv2 = nn.Conv1d(hidden_channels[0], hidden_channels[1], kernel_size=11, padding=5)
        self.bn2 = nn.BatchNorm1d(hidden_channels[1])
        
        self.conv3 = nn.Conv1d(hidden_channels[1], hidden_channels[2], kernel_size=15, padding=7)
        self.bn3 = nn.BatchNorm1d(hidden_channels[2])
    
    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.relu(self.bn3(self.conv3(x)))
        return x


class HybridEnhancerModel(nn.Module):
    """Hybrid CNN-Transformer for enhancer prediction."""
    def __init__(self, n_transformer_layers=10, n_attention_heads=8, d_model=256, 
                 dim_feedforward=1536, dropout=0.1):
        super().__init__()
        
        self.d_model = d_model
        self.n_layers = n_transformer_layers
        self.n_heads = n_attention_heads
        
        # CNN Encoder
        self.cnn_encoder = CNNEncoder(input_channels=4, hidden_channels=[128, 256, 384])
        
        # Project to d_model
        self.projection = nn.Linear(384, d_model)
        
        # Positional encoding
        self.pos_encoder = PositionalEncoding(d_model, max_len=200, dropout=dropout)
        
        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_attention_heads, dim_feedforward=dim_feedforward,
            dropout=dropout, activation='gelu', batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_transformer_layers)
        
        # Classification head
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc1 = nn.Linear(d_model, 256)
        self.dropout = nn.Dropout(dropout)
        self.fc2 = nn.Linear(256, 2)
    
    def forward(self, x):
        # CNN encoding: (batch, 4, 200) -> (batch, 384, 200)
        x = self.cnn_encoder(x)
        
        # Transpose: (batch, 384, 200) -> (batch, 200, 384)
        x = x.transpose(1, 2)
        
        # Project: (batch, 200, 384) -> (batch, 200, 256)
        x = self.projection(x)
        
        # Positional encoding
        x = self.pos_encoder(x)
        
        # Transformer: (batch, 200, 256)
        x = self.transformer(x)
        
        # Global pooling: (batch, 200, 256) -> (batch, 256)
        x = x.transpose(1, 2)
        x = self.pool(x).squeeze(-1)
        
        # Classification
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        logits = self.fc2(x)
        
        return logits
    
    def get_num_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


# Create model
model = HybridEnhancerModel(
    n_transformer_layers=CONFIG['n_transformer_layers'],
    n_attention_heads=CONFIG['n_attention_heads'],
    d_model=CONFIG['d_model'],
    dim_feedforward=CONFIG['dim_feedforward'],
    dropout=CONFIG['dropout']
).to(device)

print(f"Model created: {model.get_num_params() / 1e6:.2f}M parameters")

# Test forward pass
test_input = torch.randn(4, 4, 200).to(device)
test_output = model(test_input)
print(f"Test forward pass: input={test_input.shape}, output={test_output.shape}")

## 5. Training Pipeline

### 5.1 Dataset and DataLoader

In [ ]:
from torch.utils.data import Dataset, DataLoader, SubsetRandomSampler

class EnhancerDataset(Dataset):
    def __init__(self, X, y, species=None):
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y).long()
        self.species = torch.from_numpy(species).long() if species is not None else None
    
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


class StochasticEpochSampler:
    """Samples random subset each epoch for memory efficiency."""
    def __init__(self, train_indices, samples_per_epoch=100000, seed=42):
        self.train_indices = np.array(train_indices)
        self.samples_per_epoch = min(samples_per_epoch, len(self.train_indices))
        self.rng = np.random.RandomState(seed)
    
    def sample_epoch(self):
        return self.rng.choice(self.train_indices, size=self.samples_per_epoch, replace=False)


# Create datasets
full_dataset = EnhancerDataset(X, y, species)

# Validation loader
val_loader = DataLoader(
    full_dataset, batch_size=CONFIG['batch_size'],
    sampler=SubsetRandomSampler(val_idx),
    num_workers=CONFIG['num_workers'], pin_memory=CONFIG['pin_memory']
)

# Test loader
test_loader = DataLoader(
    full_dataset, batch_size=CONFIG['batch_size'],
    sampler=SubsetRandomSampler(test_idx),
    num_workers=CONFIG['num_workers'], pin_memory=CONFIG['pin_memory']
)

# Stochastic sampler for training
train_sampler = StochasticEpochSampler(
    train_idx, samples_per_epoch=CONFIG['samples_per_epoch']
)

print("Datasets created!")

### 5.2 Checkpointing Functions

In [ ]:
def save_checkpoint(state, checkpoint_dir, is_best=False):
    """Save training checkpoint."""
    checkpoint_dir = Path(checkpoint_dir)
    checkpoint_dir.mkdir(exist_ok=True, parents=True)
    
    torch.save(state, checkpoint_dir / 'checkpoint_latest.pt')
    print(f"   Checkpoint saved: epoch {state['epoch']}")
    
    if is_best:
        torch.save(state, checkpoint_dir / 'checkpoint_best.pt')
        print(f"   Best model updated!")
    
    if state['epoch'] % 5 == 0:
        torch.save(state, checkpoint_dir / f"checkpoint_epoch_{state['epoch']}.pt")


def load_checkpoint(checkpoint_path, model, optimizer=None, scheduler=None, scaler=None):
    """Load training checkpoint."""
    checkpoint_path = Path(checkpoint_path)
    if not checkpoint_path.exists():
        return None
    
    print(f"Loading checkpoint: {checkpoint_path.name}")
    checkpoint = torch.load(checkpoint_path, map_location=device)
    
    model.load_state_dict(checkpoint['model_state_dict'])
    if optimizer and 'optimizer_state_dict' in checkpoint:
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    if scheduler and 'scheduler_state_dict' in checkpoint:
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    if scaler and 'scaler_state_dict' in checkpoint:
        scaler.load_state_dict(checkpoint['scaler_state_dict'])
    
    print(f"   Resumed from epoch {checkpoint['epoch']}")
    return checkpoint

### 5.3 Training Loop

In [ ]:
import torch.optim as optim

def train_model(model, full_dataset, train_sampler, val_loader, device, config):
    """Complete training loop with checkpointing."""
    
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=config['learning_rate'], weight_decay=config['weight_decay'])
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=config['scheduler_factor'],
        patience=config['scheduler_patience'], min_lr=config['scheduler_min_lr']
    )
    
    use_amp = config['use_mixed_precision']
    scaler = torch.cuda.amp.GradScaler() if use_amp else None
    
    # Try to resume
    start_epoch = 0
    best_val_loss = float('inf')
    patience_counter = 0
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    
    checkpoint = load_checkpoint(
        Path(config['checkpoint_dir']) / 'checkpoint_latest.pt',
        model, optimizer, scheduler, scaler
    )
    
    if checkpoint:
        start_epoch = checkpoint['epoch'] + 1
        best_val_loss = checkpoint['best_val_loss']
        patience_counter = checkpoint['patience_counter']
        history = checkpoint['history']
    
    # Training loop
    for epoch in range(start_epoch, config['num_epochs']):
        print(f"\n{'='*60}")
        print(f"EPOCH {epoch+1}/{config['num_epochs']}")
        print(f"{'='*60}")
        
        # Sample for this epoch
        epoch_indices = train_sampler.sample_epoch()
        train_loader = DataLoader(
            full_dataset, batch_size=config['batch_size'],
            sampler=SubsetRandomSampler(epoch_indices),
            num_workers=config['num_workers'], pin_memory=config['pin_memory']
        )
        
        # Training phase
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0
        optimizer.zero_grad()
        
        pbar = tqdm(train_loader, desc="Training")
        for batch_idx, (sequences, labels) in enumerate(pbar):
            sequences, labels = sequences.to(device), labels.to(device)
            
            with torch.cuda.amp.autocast(enabled=use_amp):
                outputs = model(sequences)
                loss = criterion(outputs, labels) / config['gradient_accumulation_steps']
            
            if scaler:
                scaler.scale(loss).backward()
            else:
                loss.backward()
            
            if (batch_idx + 1) % config['gradient_accumulation_steps'] == 0:
                if scaler:
                    scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), config['max_grad_norm'])
                
                if scaler:
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    optimizer.step()
                optimizer.zero_grad()
            
            train_loss += loss.item() * config['gradient_accumulation_steps']
            _, predicted = torch.max(outputs, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()
            
            pbar.set_postfix({'loss': f'{loss.item() * config["gradient_accumulation_steps"]:.4f}'})
        
        avg_train_loss = train_loss / len(train_loader)
        train_acc = 100. * train_correct / train_total
        
        # Validation phase
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            for sequences, labels in tqdm(val_loader, desc="Validation"):
                sequences, labels = sequences.to(device), labels.to(device)
                with torch.cuda.amp.autocast(enabled=use_amp):
                    outputs = model(sequences)
                    loss = criterion(outputs, labels)
                
                val_loss += loss.item()
                _, predicted = torch.max(outputs, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()
        
        avg_val_loss = val_loss / len(val_loader)
        val_acc = 100. * val_correct / val_total
        
        scheduler.step(avg_val_loss)
        
        history['train_loss'].append(avg_train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(avg_val_loss)
        history['val_acc'].append(val_acc)
        
        print(f"\nEpoch {epoch+1} Summary:")
        print(f"   Train Loss: {avg_train_loss:.4f} | Train Acc: {train_acc:.2f}%")
        print(f"   Val Loss:   {avg_val_loss:.4f} | Val Acc:   {val_acc:.2f}%")
        
        is_best = avg_val_loss < best_val_loss
        if is_best:
            best_val_loss = avg_val_loss
            patience_counter = 0
            print("   New best model!")
        else:
            patience_counter += 1
            print(f"   No improvement ({patience_counter}/{config['early_stopping_patience']})")
        
        # Save checkpoint
        save_checkpoint({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'scaler_state_dict': scaler.state_dict() if scaler else None,
            'best_val_loss': best_val_loss,
            'patience_counter': patience_counter,
            'history': history,
        }, config['checkpoint_dir'], is_best=is_best)
        
        if patience_counter >= config['early_stopping_patience']:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break
    
    # Load best model
    load_checkpoint(Path(config['checkpoint_dir']) / 'checkpoint_best.pt', model)
    
    return history

### 5.4 Run Training

In [ ]:
# Train the model
print("Starting training...")
print(f"Checkpoints will be saved to: {CONFIG['checkpoint_dir']}")

history = train_model(
    model=model,
    full_dataset=full_dataset,
    train_sampler=train_sampler,
    val_loader=val_loader,
    device=device,
    config=CONFIG
)

print("\nTraining complete!")

### 5.5 Plot Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs = range(1, len(history['train_loss']) + 1)

# Loss
axes[0].plot(epochs, history['train_loss'], 'b-', linewidth=2, label='Train')
axes[0].plot(epochs, history['val_loss'], 'r-', linewidth=2, label='Validation')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss', fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(epochs, history['train_acc'], 'b-', linewidth=2, label='Train')
axes[1].plot(epochs, history['val_acc'], 'r-', linewidth=2, label='Validation')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Training and Validation Accuracy', fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{CONFIG['figures_dir']}/training_curves.png", dpi=300, bbox_inches='tight')
plt.show()

print(f"Final Train Acc: {history['train_acc'][-1]:.2f}%")
print(f"Final Val Acc: {history['val_acc'][-1]:.2f}%")

### 5.6 Evaluate on Test Set

In [ ]:
from sklearn.metrics import confusion_matrix, roc_auc_score

model.eval()
test_loss = 0.0
all_preds = []
all_labels = []
all_probs = []

criterion = nn.CrossEntropyLoss()

with torch.no_grad():
    for sequences, labels in tqdm(test_loader, desc="Testing"):
        sequences, labels = sequences.to(device), labels.to(device)
        outputs = model(sequences)
        loss = criterion(outputs, labels)
        
        test_loss += loss.item()
        probs = torch.softmax(outputs, dim=1)
        _, predicted = torch.max(outputs, 1)
        
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs[:, 1].cpu().numpy())

test_loss /= len(test_loader)
test_acc = 100. * (np.array(all_preds) == np.array(all_labels)).sum() / len(all_labels)
test_auc = roc_auc_score(all_labels, all_probs)
cm = confusion_matrix(all_labels, all_preds)

print(f"\nTest Results:")
print(f"   Test Loss: {test_loss:.4f}")
print(f"   Test Accuracy: {test_acc:.2f}%")
print(f"   ROC AUC: {test_auc:.3f}")
print(f"\nConfusion Matrix:")
print(f"   TN: {cm[0,0]:,} | FP: {cm[0,1]:,}")
print(f"   FN: {cm[1,0]:,} | TP: {cm[1,1]:,}")

test_results = {
    'test_loss': test_loss,
    'test_accuracy': test_acc,
    'auc': test_auc,
    'confusion_matrix': cm
}

## 6. Innovation #1: Cross-Species Evolutionary Analysis

Test repair success across vertebrate species.

In [ ]:
def damage_sequence(sequence, damage_fraction=0.1, seed=None):
    """Randomly damage DNA sequence."""
    if seed is not None:
        np.random.seed(seed)
    
    sequence = sequence.copy()
    seq_length = sequence.shape[1]
    n_damage = int(seq_length * damage_fraction)
    damaged_positions = np.random.choice(seq_length, size=n_damage, replace=False)
    
    for pos in damaged_positions:
        current_nt = np.argmax(sequence[:, pos])
        possible = [0, 1, 2, 3]
        possible.remove(current_nt)
        new_nt = np.random.choice(possible)
        sequence[:, pos] = 0
        sequence[new_nt, pos] = 1
    
    return sequence, damaged_positions


def attempt_repair(damaged_sequence, model, device, max_iterations=30, 
                   learning_rate=0.01, target_prob=0.9):
    """Attempt to repair damaged sequence using gradient descent."""
    model.eval()
    
    seq_tensor = torch.from_numpy(damaged_sequence).float().unsqueeze(0).to(device)
    seq_tensor.requires_grad_(True)
    
    trajectory = []
    
    for iteration in range(max_iterations):
        outputs = model(seq_tensor)
        probs = torch.softmax(outputs, dim=1)
        enhancer_prob = probs[0, 1]
        
        trajectory.append(enhancer_prob.item())
        
        if enhancer_prob.item() >= target_prob:
            return True, enhancer_prob.item(), trajectory
        
        loss = -torch.log(enhancer_prob + 1e-8)
        loss.backward()
        
        with torch.no_grad():
            seq_tensor.data -= learning_rate * seq_tensor.grad
            seq_data = seq_tensor.squeeze(0)
            max_indices = torch.argmax(seq_data, dim=0)
            new_seq = torch.zeros_like(seq_data)
            for pos in range(200):
                new_seq[max_indices[pos], pos] = 1.0
            seq_tensor.data = new_seq.unsqueeze(0)
            if seq_tensor.grad is not None:
                seq_tensor.grad.zero_()
    
    return False, trajectory[-1] if trajectory else 0.0, trajectory

In [ ]:
# Run cross-species analysis
print("Running cross-species repair analysis...")

species_names = ['Human', 'Mouse', 'Zebrafish', 'Chicken']
cross_species_results = {}

for species_id, species_name in enumerate(species_names):
    # Get test enhancers for this species
    species_test_mask = (species[test_idx] == species_id) & (y[test_idx] == 1)
    species_seqs = X[test_idx][species_test_mask]
    
    if len(species_seqs) == 0:
        print(f"No test sequences for {species_name}")
        continue
    
    n_test = min(CROSS_SPECIES_CONFIG['n_sequences_per_species'], len(species_seqs))
    test_seqs = species_seqs[:n_test]
    
    successes = []
    print(f"\n{species_name}: Testing {n_test} sequences...")
    
    for seq in tqdm(test_seqs, desc=f"  {species_name}"):
        damaged, _ = damage_sequence(seq, damage_fraction=CROSS_SPECIES_CONFIG['damage_fraction'])
        success, _, _ = attempt_repair(
            damaged, model, device,
            max_iterations=CROSS_SPECIES_CONFIG['max_repair_iterations'],
            learning_rate=CROSS_SPECIES_CONFIG['repair_learning_rate'],
            target_prob=CROSS_SPECIES_CONFIG['target_probability']
        )
        successes.append(success)
    
    repair_rate = np.mean(successes) * 100
    cross_species_results[species_name] = {
        'repair_rate': repair_rate,
        'n_tested': n_test,
        'successes': sum(successes)
    }
    print(f"  Repair rate: {repair_rate:.1f}%")

print("\nCross-species analysis complete!")

In [ ]:
# Visualize cross-species results
fig, ax = plt.subplots(figsize=(10, 6))

species = list(cross_species_results.keys())
repair_rates = [cross_species_results[s]['repair_rate'] for s in species]
colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12'][:len(species)]

bars = ax.bar(species, repair_rates, color=colors, alpha=0.8, edgecolor='black')

for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.1f}%', ha='center', va='bottom', fontsize=12, fontweight='bold')

ax.set_ylabel('Repair Success Rate (%)', fontsize=13)
ax.set_title('Cross-Species Repair Analysis: Evolutionary Conservation', fontsize=14, fontweight='bold')
ax.set_ylim(0, 100)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(f"{CONFIG['figures_dir']}/cross_species_repair.png", dpi=300, bbox_inches='tight')
plt.show()

# Save results
with open(f"{CONFIG['results_dir']}/cross_species/results.pkl", 'wb') as f:
    pickle.dump(cross_species_results, f)

## 7. Innovation #2: Hierarchical Grammar Discovery

Discover which motifs work synergistically vs independently.

In [ ]:
# Extract important regions using gradient importance
print("Extracting motif regions...")

# Get enhancer test sequences
enhancer_test_mask = y[test_idx] == 1
enhancer_seqs = X[test_idx][enhancer_test_mask][:GRAMMAR_CONFIG['n_sequences_for_motifs']]

all_importances = []

for seq in tqdm(enhancer_seqs, desc="Computing importance"):
    seq_tensor = torch.from_numpy(seq).float().unsqueeze(0).to(device)
    seq_tensor.requires_grad_(True)
    
    outputs = model(seq_tensor)
    enhancer_prob = torch.softmax(outputs, dim=1)[0, 1]
    enhancer_prob.backward()
    
    grad = seq_tensor.grad.abs().sum(dim=1).squeeze().detach().cpu().numpy()
    all_importances.append(grad)

avg_importance = np.mean(all_importances, axis=0)

# Find peaks
from scipy.signal import find_peaks
threshold = np.percentile(avg_importance, GRAMMAR_CONFIG['importance_percentile'])
peaks, _ = find_peaks(avg_importance, height=threshold, distance=6)

# Define motif regions
motif_regions = [(max(0, p-4), min(200, p+5)) for p in peaks]
print(f"Found {len(motif_regions)} candidate motif regions")

In [ ]:
# Test pairwise interactions
def damage_region(sequence, start, end):
    """Damage specific region."""
    damaged = sequence.copy()
    for pos in range(start, min(end, sequence.shape[1])):
        current = np.argmax(sequence[:, pos])
        possible = [0, 1, 2, 3]
        possible.remove(current)
        damaged[:, pos] = 0
        damaged[np.random.choice(possible), pos] = 1
    return damaged

print("Discovering grammar rules...")
interactions = []
n_motifs = len(motif_regions)

if n_motifs >= 2:
    pairs_tested = set()
    
    for _ in tqdm(range(min(GRAMMAR_CONFIG['n_pairs_to_test'], n_motifs*(n_motifs-1)//2)), desc="Testing pairs"):
        # Get unique pair
        attempts = 0
        while attempts < 100:
            i, j = np.random.choice(n_motifs, size=2, replace=False)
            pair_key = (min(i, j), max(i, j))
            if pair_key not in pairs_tested:
                pairs_tested.add(pair_key)
                break
            attempts += 1
        
        if attempts >= 100:
            break
        
        motif1, motif2 = motif_regions[i], motif_regions[j]
        
        results = []
        for seq_idx in range(min(GRAMMAR_CONFIG['n_sequences_per_pair'], len(enhancer_seqs))):
            seq = enhancer_seqs[seq_idx]
            
            dam1 = damage_region(seq, motif1[0], motif1[1])
            r1, _, _ = attempt_repair(dam1, model, device)
            
            dam2 = damage_region(seq, motif2[0], motif2[1])
            r2, _, _ = attempt_repair(dam2, model, device)
            
            dam_both = damage_region(dam1, motif2[0], motif2[1])
            r_both, _, _ = attempt_repair(dam_both, model, device)
            
            results.append((r1, r2, r_both))
        
        avg_r1 = np.mean([r[0] for r in results])
        avg_r2 = np.mean([r[1] for r in results])
        avg_r_both = np.mean([r[2] for r in results])
        expected = avg_r1 * avg_r2
        
        if expected > 0:
            ratio = avg_r_both / expected
        else:
            ratio = 1.0
        
        if ratio < 0.5:
            interaction_type = "SYNERGY"
        elif ratio > 1.5:
            interaction_type = "REDUNDANT"
        else:
            interaction_type = "INDEPENDENT"
        
        interactions.append({
            'motif1': i, 'motif2': j,
            'repair1': avg_r1, 'repair2': avg_r2, 'repair_both': avg_r_both,
            'interaction': interaction_type
        })

# Summarize
synergistic = [x for x in interactions if x['interaction'] == 'SYNERGY']
redundant = [x for x in interactions if x['interaction'] == 'REDUNDANT']
independent = [x for x in interactions if x['interaction'] == 'INDEPENDENT']

print(f"\nGrammar Discovery Results:")
print(f"   Synergistic: {len(synergistic)} ({len(synergistic)/max(1,len(interactions))*100:.1f}%)")
print(f"   Redundant: {len(redundant)} ({len(redundant)/max(1,len(interactions))*100:.1f}%)")
print(f"   Independent: {len(independent)} ({len(independent)/max(1,len(interactions))*100:.1f}%)")

In [ ]:
# Visualize grammar results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Importance profile
ax = axes[0]
ax.plot(avg_importance, linewidth=2, color='darkblue')
ax.fill_between(range(200), 0, avg_importance, alpha=0.3, color='skyblue')
for start, end in motif_regions:
    ax.axvspan(start, end, alpha=0.2, color='red')
ax.set_xlabel('Position (bp)')
ax.set_ylabel('Importance')
ax.set_title('Position Importance Profile', fontweight='bold')
ax.grid(True, alpha=0.3)

# Interaction pie chart
ax = axes[1]
counts = [len(synergistic), len(redundant), len(independent)]
if sum(counts) > 0:
    ax.pie(counts, labels=['Synergistic', 'Redundant', 'Independent'], 
           autopct='%1.1f%%', colors=['#e74c3c', '#f39c12', '#95a5a6'])
ax.set_title('Grammar Rule Distribution', fontweight='bold')

plt.tight_layout()
plt.savefig(f"{CONFIG['figures_dir']}/grammar_discovery.png", dpi=300, bbox_inches='tight')
plt.show()

# Save results
with open(f"{CONFIG['results_dir']}/grammar/results.pkl", 'wb') as f:
    pickle.dump({'motif_regions': motif_regions, 'interactions': interactions}, f)

## 8. Innovation #3: Generative Validation

Generate functional enhancers from random DNA.

In [ ]:
def generate_enhancer_from_random(model, device, max_iterations=500, target_prob=0.80, lr=0.05):
    """Generate enhancer from random DNA."""
    model.eval()
    
    # Random initial sequence
    random_seq = np.zeros((4, 200), dtype=np.float32)
    for i in range(200):
        random_seq[np.random.randint(0, 4), i] = 1.0
    
    seq_tensor = torch.from_numpy(random_seq).float().unsqueeze(0).to(device)
    seq_tensor.requires_grad_(True)
    
    trajectory = []
    
    for _ in range(max_iterations):
        outputs = model(seq_tensor)
        probs = torch.softmax(outputs, dim=1)
        enhancer_prob = probs[0, 1]
        trajectory.append(enhancer_prob.item())
        
        if enhancer_prob.item() >= target_prob:
            return seq_tensor.squeeze(0).detach().cpu().numpy(), trajectory, True
        
        loss = -torch.log(enhancer_prob + 1e-8)
        loss.backward()
        
        with torch.no_grad():
            seq_tensor.data -= lr * seq_tensor.grad
            seq_data = seq_tensor.squeeze(0)
            max_indices = torch.argmax(seq_data, dim=0)
            new_seq = torch.zeros_like(seq_data)
            for pos in range(200):
                new_seq[max_indices[pos], pos] = 1.0
            seq_tensor.data = new_seq.unsqueeze(0)
            if seq_tensor.grad is not None:
                seq_tensor.grad.zero_()
    
    return seq_tensor.squeeze(0).detach().cpu().numpy(), trajectory, False


# Generate enhancers
print("Generating synthetic enhancers...")

generated_seqs = []
trajectories = []

for _ in tqdm(range(GENERATIVE_CONFIG['n_generation_attempts']), desc="Generating"):
    seq, traj, success = generate_enhancer_from_random(
        model, device,
        max_iterations=GENERATIVE_CONFIG['max_iterations'],
        target_prob=GENERATIVE_CONFIG['target_probability'],
        lr=GENERATIVE_CONFIG['learning_rate']
    )
    if success:
        generated_seqs.append(seq)
        trajectories.append(traj)

success_rate = len(generated_seqs) / GENERATIVE_CONFIG['n_generation_attempts'] * 100
print(f"\nGenerated {len(generated_seqs)} enhancers ({success_rate:.1f}% success rate)")

In [ ]:
# Validate generated sequences
from scipy import stats

known_motifs = ['TATAAA', 'CACGTG', 'GGGCGG', 'CCAAT']

def calc_gc(seq_str):
    return (seq_str.count('G') + seq_str.count('C')) / len(seq_str)

def count_motifs(seq_str, motifs):
    return sum(seq_str.count(m) for m in motifs)

# Analyze generated
gen_gc = [calc_gc(decode_sequence(s)) for s in generated_seqs]
gen_motifs = [count_motifs(decode_sequence(s), known_motifs) for s in generated_seqs]

# Analyze real
real_seqs = X[test_idx][y[test_idx] == 1][:1000]
real_gc = [calc_gc(decode_sequence(s)) for s in real_seqs]
real_motifs = [count_motifs(decode_sequence(s), known_motifs) for s in real_seqs]

# Statistical tests
if len(gen_gc) > 0:
    gc_pvalue = stats.ttest_ind(gen_gc, real_gc).pvalue
    motif_pvalue = stats.ttest_ind(gen_motifs, real_motifs).pvalue
    
    print(f"\nValidation Results:")
    print(f"GC Content:")
    print(f"   Generated: {np.mean(gen_gc):.3f} +/- {np.std(gen_gc):.3f}")
    print(f"   Real: {np.mean(real_gc):.3f} +/- {np.std(real_gc):.3f}")
    print(f"   p-value: {gc_pvalue:.4f} {'(similar)' if gc_pvalue > 0.05 else '(different)'}")
    
    print(f"\nMotif Count:")
    print(f"   Generated: {np.mean(gen_motifs):.2f} +/- {np.std(gen_motifs):.2f}")
    print(f"   Real: {np.mean(real_motifs):.2f} +/- {np.std(real_motifs):.2f}")
    print(f"   p-value: {motif_pvalue:.4f} {'(similar)' if motif_pvalue > 0.05 else '(different)'}")
    
    validation_results = {
        'generated_gc': gen_gc, 'real_gc': real_gc,
        'generated_motifs': gen_motifs, 'real_motifs': real_motifs,
        'gc_pvalue': gc_pvalue, 'motif_pvalue': motif_pvalue,
        'success_rate': success_rate
    }
else:
    print("No sequences generated - cannot validate")
    validation_results = {'success_rate': 0}

In [ ]:
# Visualize generative results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Trajectories
ax = axes[0]
for traj in trajectories[:30]:
    ax.plot(traj, alpha=0.2, color='blue')
if trajectories:
    max_len = max(len(t) for t in trajectories)
    avg_traj = [np.mean([t[i] for t in trajectories if len(t) > i]) for i in range(max_len)]
    ax.plot(avg_traj, color='red', linewidth=3, label='Average')
ax.axhline(y=0.80, color='green', linestyle='--', linewidth=2, label='Target')
ax.set_xlabel('Iteration')
ax.set_ylabel('Enhancer Probability')
ax.set_title('Generation Trajectories', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1)

# GC comparison
ax = axes[1]
if len(gen_gc) > 0:
    ax.hist(real_gc, bins=30, alpha=0.6, label='Real', color='blue', edgecolor='black')
    ax.hist(gen_gc, bins=30, alpha=0.6, label='Generated', color='red', edgecolor='black')
    ax.legend()
ax.set_xlabel('GC Content')
ax.set_ylabel('Count')
ax.set_title('GC Content Distribution', fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(f"{CONFIG['figures_dir']}/generative_validation.png", dpi=300, bbox_inches='tight')
plt.show()

# Save results
with open(f"{CONFIG['results_dir']}/generative/results.pkl", 'wb') as f:
    pickle.dump({'generated_seqs': generated_seqs, 'trajectories': trajectories, 
                 'validation': validation_results}, f)

## 9. Comprehensive Summary Figure

In [ ]:
# Create comprehensive summary figure
fig = plt.figure(figsize=(16, 12))
gs = fig.add_gridspec(3, 3, hspace=0.35, wspace=0.3)

# Panel 1: Training curves
ax1 = fig.add_subplot(gs[0, 0:2])
epochs = range(1, len(history['train_loss']) + 1)
ax1.plot(epochs, history['train_loss'], 'b-', linewidth=2, label='Train')
ax1.plot(epochs, history['val_loss'], 'r-', linewidth=2, label='Val')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training Progress', fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Panel 2: Performance
ax2 = fig.add_subplot(gs[0, 2])
metrics = ['Train', 'Val', 'Test']
values = [history['train_acc'][-1], history['val_acc'][-1], test_results['test_accuracy']]
colors = ['#3498db', '#2ecc71', '#e74c3c']
bars = ax2.bar(metrics, values, color=colors, alpha=0.8, edgecolor='black')
for bar in bars:
    ax2.text(bar.get_x() + bar.get_width()/2., bar.get_height(),
            f'{bar.get_height():.1f}%', ha='center', va='bottom')
ax2.set_ylim(0, 100)
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Model Performance', fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')

# Panel 3: Cross-species
ax3 = fig.add_subplot(gs[1, 0])
species_list = list(cross_species_results.keys())
rates = [cross_species_results[s]['repair_rate'] for s in species_list]
ax3.bar(species_list, rates, color=['#e74c3c', '#3498db', '#2ecc71', '#f39c12'][:len(species_list)], 
        alpha=0.8, edgecolor='black')
ax3.set_ylabel('Repair Rate (%)')
ax3.set_title('Cross-Species', fontweight='bold')
ax3.tick_params(axis='x', rotation=45)
ax3.grid(True, alpha=0.3, axis='y')

# Panel 4: Grammar
ax4 = fig.add_subplot(gs[1, 1])
counts = [len(synergistic), len(redundant), len(independent)]
if sum(counts) > 0:
    ax4.pie(counts, labels=['Synergy', 'Redundant', 'Indep'], autopct='%1.1f%%',
            colors=['#e74c3c', '#f39c12', '#95a5a6'])
ax4.set_title('Grammar Rules', fontweight='bold')

# Panel 5: Generation
ax5 = fig.add_subplot(gs[1, 2])
gen_metrics = ['Success', 'GC Match', 'Motif Match']
gen_values = [
    validation_results.get('success_rate', 0),
    100 if validation_results.get('gc_pvalue', 0) > 0.05 else 50,
    100 if validation_results.get('motif_pvalue', 0) > 0.05 else 50
]
ax5.bar(gen_metrics, gen_values, color=['#9b59b6', '#1abc9c', '#e67e22'], alpha=0.8, edgecolor='black')
ax5.set_ylim(0, 100)
ax5.set_ylabel('Score')
ax5.set_title('Generation', fontweight='bold')
ax5.tick_params(axis='x', rotation=45)
ax5.grid(True, alpha=0.3, axis='y')

# Panel 6: Summary text
ax6 = fig.add_subplot(gs[2, :])
ax6.axis('off')
summary_text = f"""
RESEARCH SUMMARY
================
Model: Hybrid CNN-Transformer ({model.get_num_params()/1e6:.1f}M parameters)
Test Accuracy: {test_results['test_accuracy']:.1f}%  |  ROC AUC: {test_results['auc']:.3f}

Innovation #1 - Cross-Species Conservation:
  Tested repair across {len(cross_species_results)} vertebrate species
  Average repair rate: {np.mean([r['repair_rate'] for r in cross_species_results.values()]):.1f}%

Innovation #2 - Grammar Discovery:
  Found {len(motif_regions)} motif regions, tested {len(interactions)} pairs
  Synergistic: {len(synergistic)}  |  Redundant: {len(redundant)}  |  Independent: {len(independent)}

Innovation #3 - Generative Validation:
  Generation success rate: {validation_results.get('success_rate', 0):.1f}%
  GC content similar to real enhancers: {'Yes' if validation_results.get('gc_pvalue', 0) > 0.05 else 'No'}
"""
ax6.text(0.1, 0.9, summary_text, transform=ax6.transAxes, fontsize=11, 
         verticalalignment='top', family='monospace',
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.suptitle('ENHANCER REPAIR RESEARCH: COMPREHENSIVE ANALYSIS', fontsize=16, fontweight='bold', y=0.995)
plt.savefig(f"{CONFIG['figures_dir']}/comprehensive_summary.png", dpi=300, bbox_inches='tight')
plt.show()

## 10. Save Final Results

In [ ]:
# Save model
torch.save({
    'model_state_dict': model.state_dict(),
    'config': CONFIG,
    'history': history,
    'test_results': test_results
}, f"{CONFIG['project_dir']}/models/final_model.pt")

# Save all results
all_results = {
    'training_history': history,
    'test_results': test_results,
    'cross_species_results': cross_species_results,
    'grammar_results': {
        'motif_regions': motif_regions,
        'interactions': interactions,
        'synergistic': len(synergistic),
        'redundant': len(redundant),
        'independent': len(independent)
    },
    'generative_results': validation_results
}

with open(f"{CONFIG['project_dir']}/results/all_results.pkl", 'wb') as f:
    pickle.dump(all_results, f)

print(f"\nAll results saved to {CONFIG['project_dir']}")
print("\n" + "="*60)
print("RESEARCH COMPLETE!")
print("="*60)

---

## Notes for Resuming After Colab Timeout

If Colab disconnects, your checkpoints are saved to Google Drive.

To resume:
1. Re-run cells 1-4 (Environment setup)
2. Skip data generation (cell 3.2-3.4) if data already exists
3. Re-create model (cell 4.1)
4. Run training - it will auto-resume from checkpoint

```python
# Quick resume check
import os
checkpoint_path = f"{PROJECT_DIR}/checkpoints/checkpoint_latest.pt"
if os.path.exists(checkpoint_path):
    print(f"Checkpoint found! Training will resume.")
else:
    print("No checkpoint - starting fresh.")
```